In [1]:
# ============================================================
# FINAL MODELS — BOOTSTRAP 95% CI FOR PERFORMANCE METRICS
# PCA TRAIN-ONLY VERSION
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import confusion_matrix, accuracy_score, balanced_accuracy_score

RANDOM_STATE = 42
N_BOOT = 5000
ALPHA = 0.05

# ------------------------------------------------------------
# Paths to final prediction files
# ------------------------------------------------------------

PREDICTION_FILES = {
    "victimization": Path("./final_victim/DT_victim_PCA_trainonly_TEST/predictions_with_probs.csv"),
    "perpetration": Path("./final_perpetrator/content/perpetrator_v3/predictions_with_probs.csv"),
    "overlap": Path("./final_overlap/overlap_final/overlap_LOGREG_SW_pos1p5_PCA_trainonly_FINAL_PCA095_thr0p5_seed42/outputs/predictions_with_probs.csv"),
}

OUTPUT_DIR = Path("./final_bootstrap_CI_PCA_trainonly")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def detect_col(df, candidates):
    cols_lower = {str(c).lower(): c for c in df.columns}

    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]

    for c in df.columns:
        cl = str(c).lower()
        if any(cand.lower() in cl for cand in candidates):
            return c

    return None


def binary_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan
    f1 = 2 * ppv * recall / (ppv + recall) if (ppv + recall) > 0 else np.nan

    return {
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
        "recall_sensitivity": recall,
        "specificity": specificity,
        "precision_ppv": ppv,
        "npv": npv,
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1_positive": f1,
        "accuracy": accuracy_score(y_true, y_pred),
        "support": int(len(y_true)),
        "positive_support": int(np.sum(y_true == 1)),
        "negative_support": int(np.sum(y_true == 0)),
    }


def bootstrap_ci(y_true, y_pred, n_boot=5000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)

    boot_rows = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        yp = y_pred[idx]

        # Skip pathological bootstrap samples with only one class
        if len(np.unique(yt)) < 2:
            continue

        m = binary_metrics(yt, yp)
        boot_rows.append(m)

    boot_df = pd.DataFrame(boot_rows)

    ci_rows = []
    metric_names = [
        "recall_sensitivity",
        "specificity",
        "precision_ppv",
        "npv",
        "balanced_accuracy",
        "f1_positive",
        "accuracy",
    ]

    point = binary_metrics(y_true, y_pred)

    for metric in metric_names:
        values = boot_df[metric].dropna().to_numpy()

        ci_rows.append({
            "metric": metric,
            "point": point[metric],
            "ci_lower": np.percentile(values, 100 * alpha / 2),
            "ci_upper": np.percentile(values, 100 * (1 - alpha / 2)),
            "n_boot_valid": len(values),
        })

    return pd.DataFrame(ci_rows), boot_df, point


def load_predictions(path, outcome):
    path = Path(path)
    df = pd.read_csv(path)

    print("\n======================================")
    print("Outcome:", outcome)
    print("Path:", path)
    print("Exists:", path.exists())
    print("Columns:", list(df.columns))
    print(df.head())

    y_true_col = detect_col(df, [
        f"y_true_{outcome}",
        "y_true_victim",
        "y_true_perp",
        "y_true_perpetration",
        "y_true_overlap",
        "y_true_intersect",
        "y_true",
        "target"
    ])

    y_pred_col = detect_col(df, [
        f"y_pred_{outcome}",
        "y_pred_victim",
        "y_pred_perp",
        "y_pred_perpetration",
        "y_pred_overlap",
        "y_pred_intersect",
        "y_pred",
        "prediction"
    ])

    if y_true_col is None or y_pred_col is None:
        raise ValueError(
            f"Could not detect y_true/y_pred columns for {outcome}. "
            f"Columns found: {list(df.columns)}"
        )

    y_true = df[y_true_col].astype(int).to_numpy()
    y_pred = df[y_pred_col].astype(int).to_numpy()

    print("Detected y_true:", y_true_col)
    print("Detected y_pred:", y_pred_col)
    print("n:", len(y_true), "positives:", y_true.sum(), "negatives:", len(y_true) - y_true.sum())

    return df, y_true, y_pred


# ------------------------------------------------------------
# Run bootstrap for all final models
# ------------------------------------------------------------

all_ci = []
all_points = []

for outcome, path in PREDICTION_FILES.items():
    df, y_true, y_pred = load_predictions(path, outcome)

    ci_df, boot_df, point = bootstrap_ci(
        y_true=y_true,
        y_pred=y_pred,
        n_boot=N_BOOT,
        alpha=ALPHA,
        seed=RANDOM_STATE
    )

    ci_df.insert(0, "outcome", outcome)

    point_row = {
        "outcome": outcome,
        **point
    }

    all_ci.append(ci_df)
    all_points.append(point_row)

    ci_df.to_csv(OUTPUT_DIR / f"{outcome}_bootstrap_CI.csv", index=False)
    boot_df.to_csv(OUTPUT_DIR / f"{outcome}_bootstrap_raw.csv", index=False)

final_ci_df = pd.concat(all_ci, ignore_index=True)
final_points_df = pd.DataFrame(all_points)

final_ci_df.to_csv(OUTPUT_DIR / "final_models_bootstrap_CI_all.csv", index=False)
final_points_df.to_csv(OUTPUT_DIR / "final_models_point_metrics_all.csv", index=False)


# ------------------------------------------------------------
# Pretty table for manuscript
# ------------------------------------------------------------

pretty = final_ci_df.copy()

pretty["point_pct"] = pretty["point"] * 100
pretty["ci_lower_pct"] = pretty["ci_lower"] * 100
pretty["ci_upper_pct"] = pretty["ci_upper"] * 100

pretty["formatted"] = pretty.apply(
    lambda r: f"{r['point_pct']:.1f}% ({r['ci_lower_pct']:.1f}–{r['ci_upper_pct']:.1f})",
    axis=1
)

pretty_table = pretty.pivot(
    index="outcome",
    columns="metric",
    values="formatted"
).reset_index()

# Reorder columns
ordered_cols = [
    "outcome",
    "recall_sensitivity",
    "specificity",
    "precision_ppv",
    "npv",
    "balanced_accuracy",
    "f1_positive",
    "accuracy",
]

pretty_table = pretty_table[ordered_cols]

pretty_table.to_csv(OUTPUT_DIR / "final_models_bootstrap_CI_pretty_table.csv", index=False)

print("\n=== POINT METRICS ===")
print(final_points_df.to_string(index=False))

print("\n=== BOOTSTRAP 95% CI — PRETTY TABLE ===")
print(pretty_table.to_string(index=False))

print("\nSaved outputs to:")
print(OUTPUT_DIR.resolve())


Outcome: victimization
Path: final_victim/DT_victim_PCA_trainonly_TEST/predictions_with_probs.csv
Exists: True
Columns: ['idx_original', 'y_true_victim', 'y_pred_victim', 'y_prob_victim']
   idx_original  y_true_victim  y_pred_victim  y_prob_victim
0          3001              1              0       0.297940
1          1694              0              0       0.297940
2           677              0              0       0.297940
3           840              0              1       0.782477
4          1903              0              1       0.509393
Detected y_true: y_true_victim
Detected y_pred: y_pred_victim
n: 942 positives: 465 negatives: 477

Outcome: perpetration
Path: final_perpetrator/content/perpetrator_v3/predictions_with_probs.csv
Exists: True
Columns: ['idx_original', 'y_true_perp', 'y_pred_perp', 'y_prob_perp']
   idx_original  y_true_perp  y_pred_perp  y_prob_perp
0             1            1            1     0.554482
1             2            0            1     0.636366


In [2]:
pretty_path = "./final_bootstrap_CI_PCA_trainonly/final_models_bootstrap_CI_pretty_table.csv"

pretty_table = pd.read_csv(pretty_path)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)

print(pretty_table.to_string(index=False))

      outcome recall_sensitivity       specificity     precision_ppv               npv balanced_accuracy       f1_positive          accuracy
      overlap  80.9% (75.1–86.5) 53.5% (50.1–57.1) 28.9% (24.9–33.0) 92.3% (89.8–94.7) 67.2% (63.9–70.5) 42.5% (37.8–47.3) 58.7% (55.6–61.9)
 perpetration  91.4% (87.5–95.0) 30.1% (26.8–33.5) 28.6% (25.2–31.9) 91.9% (88.3–95.3) 60.7% (58.2–63.2) 43.6% (39.4–47.5) 44.5% (41.3–47.7)
victimization  87.1% (83.8–90.1) 31.0% (26.9–35.2) 55.2% (51.5–58.7) 71.2% (64.7–77.3) 59.1% (56.4–61.6) 67.6% (64.4–70.5) 58.7% (55.5–61.8)
